In [1]:
import os
import time
from datetime import datetime
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm  
from torchinfo import summary

# --- Project-specific imports ---
import config as cfg
import utils.data as data_utils
from utils.model import get_model, save_model_weights
from utils.evaluate import EvalClassification

print("Imports complete. Ready to set up the training run.\n")

train_dataloader, val_dataloader, test_dataloader = data_utils.get_3_dataloaders()

/home/takayuki/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Imports complete. Ready to set up the training run.

 Total dataset size 	: 221
 Train dataset size 	: 132
 Val dataset size 	: 44
 Test dataset size 	: 45



## Load model

In [2]:
model = get_model()

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

trainable_params = [p for p in model.parameters() if p.requires_grad]
print("Number of trainable parameters: ", sum(p.numel() for p in trainable_params))

summary(model, input_size=(1, 3, 224, 224))

Number of trainable parameters:  66381


Layer (type:depth-idx)                                            Output Shape              Param #
MLPModel                                                          [1, 21]                   --
├─ConvNeXt: 1-1                                                   [1, 768]                  --
│    └─Sequential: 2-1                                            [1, 96, 56, 56]           --
│    │    └─Conv2d: 3-1                                           [1, 96, 56, 56]           (4,704)
│    │    └─LayerNorm2d: 3-2                                      [1, 96, 56, 56]           (192)
│    └─Sequential: 2-2                                            [1, 768, 7, 7]            --
│    │    └─ConvNeXtStage: 3-3                                    [1, 96, 56, 56]           (239,904)
│    │    └─ConvNeXtStage: 3-4                                    [1, 192, 28, 28]          (996,288)
│    │    └─ConvNeXtStage: 3-5                                    [1, 384, 14, 14]          (11,137,152)
│    │    └─C

## Training Loop
- Tensorboard logging
- Save model checkpoints
- Save best models

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device != torch.device("cuda"):
    raise RuntimeError("This script requires a CUDA-enabled GPU for training.")
    # You can comment out this line if you want to run on CPU for testing purposes.
    
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.to(device)
print(f"Using device: {device}, {torch.cuda.get_device_name(device) if device.type == 'cuda' else 'CPU'}")

Using device: cuda, NVIDIA RTX A500 Laptop GPU


In [ ]:
criterion = nn.CrossEntropyLoss()
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = torch.optim.AdamW(trainable_params, lr=cfg.lr_initial, weight_decay=cfg.weight_decay)

if cfg.lr_scheme == 'cosine':
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.lr_cosine_tmax_epochs, eta_min=cfg.lr_cosine_minimum)
elif cfg.lr_scheme == 'step':
    scheduler = StepLR(optimizer, step_size=cfg.lr_step_size, gamma=cfg.lr_step_gamma)
else:
    print(f"Unsupported learning rate scheme: {cfg.lr_scheme}. Using Fixed LR.")
    scheduler = None

now = datetime.now()
timestamp = now.strftime("%m-%d_%H-%M--%S")

# Save a copy of the configuration file
config_file_name = f"config_{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}.py"
cfg.save_this_config(config_file_name)

hparams_dict = {
    "a_run_name": cfg.training_run_name,
    "data_directory": cfg.main_data_dir,
    "train_dataset_percent": cfg.train_percent,
    "val_dataset_percent": cfg.val_percent,
    "test_dataset_percent": cfg.test_percent,
    "model_architecture": cfg.train_timm_model_name,
    "classification_head": cfg.classification_head_type,
    "num_trainable_parameters": sum(p.numel() for p in trainable_params),
    "optimizer": optimizer.__class__.__name__,
    "criterion": criterion.__class__.__name__,
    "scheduler": scheduler.__class__.__name__,
    "num_epochs": cfg.num_epochs,
    "batch_size": cfg.batch_size,
    "learning_rate_scheme": cfg.lr_scheme,
    "initial_learning_rate": cfg.lr_initial,
    "final_learning_rate": cfg.lr_cosine_minimum,
    "weight_decay": cfg.weight_decay,
    "random_seed": cfg.random_seed,
    "timestamp": timestamp,
    "config_file_path": os.path.join(cfg.saved_configs_path, config_file_name)
}

writer = SummaryWriter(log_dir=os.path.join(cfg.base_logs_path, f"{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}"))

for key, value in hparams_dict.items():
    writer.add_text(key, str(value))

# Training loop
best_val_loss = float('inf') 
best_val_accuracy = 0.0
best_val_epoch = 0

print("Starting training...")
start_time = time.time()
for epoch in range(cfg.num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{cfg.num_epochs}", unit="batch"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = running_loss / total
    train_accuracy = correct / total
    
    lr = scheduler.get_last_lr()[0] if scheduler else cfg.lr_initial
    print(f"Training Loss: {train_loss:.4f}, Training Accuracy: {train_accuracy*100:.4f}% at LR: {lr:.6f}")
    
    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_accuracy, epoch)
    
    # Validation phase
    if (epoch + 1) % cfg.validation_interval == 0:
        model.eval()  
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():  
            
            for val_images, val_labels in val_dataloader:
                val_images, val_labels = val_images.to(device), val_labels.to(device)
                
                val_outputs = model(val_images)
                val_loss = criterion(val_outputs, val_labels)
                
                val_running_loss += val_loss.item() * val_images.size(0)
                _, val_predicted = torch.max(val_outputs.data, 1)
                val_total += val_labels.size(0)
                val_correct += (val_predicted == val_labels).sum().item()
        
        val_loss = val_running_loss / val_total
        val_accuracy = val_correct / val_total

        print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy*100:.4f}%")
        
        writer.add_scalar('Loss/validation', val_loss, epoch)
        writer.add_scalar('Accuracy/validation', val_accuracy, epoch)
        
        # best model
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            print(f"Best Validation Accuracy: {best_val_accuracy*100:.4f}% at Epoch {epoch + 1}")
            writer.add_scalar('Best_Validation_Accuracy', best_val_accuracy, epoch)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_epoch = epoch + 1
            
            # Delete previous best model if exists
            best_model_name = f"best_model_{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}.pth"
            best_model_file_path = os.path.join(cfg.models_path, best_model_name)
            
            if os.path.exists(best_model_file_path):
                os.remove(best_model_file_path)
            
            # Save the new one
            save_model_weights(model, best_model_file_path, verbose=False) 
            print(f"Saved new best model: {best_model_file_path}")
            print(f"Best Validation Loss: {best_val_loss:.4f} at Epoch {epoch + 1}")
            writer.add_scalar('Best_Validation_Loss', best_val_loss, epoch)
    
    # Update learning rate
    if scheduler:
        scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    writer.add_scalar('Learning_Rate', current_lr, epoch)
    
    # Checkpoint saving phase
    if (epoch + 1) % cfg.checkpoint_interval == 0 or epoch == cfg.num_epochs - 1:
        checkpoint_name = f"checkpoint_{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}_epoch_{epoch + 1}.pth"
        checkpoint_file_path = os.path.join(cfg.checkpoint_path, checkpoint_name)
        
        torch.save({
            'epoch': epoch + 1,
            'model_config': model.config,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss, 
            'log_dir': writer.log_dir,
            'best_val_accuracy': best_val_accuracy,
            'best_val_loss': best_val_loss,
            'best_val_epoch': best_val_epoch,
            'hparams': hparams_dict
        }, checkpoint_file_path)
        
        print(f"Saved checkpoint: {checkpoint_file_path}")

print("\nTraining complete.")
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Best Validation Accuracy: {best_val_accuracy*100:.4f}% at Epoch {best_val_epoch}")
print(f"Final Training Accuracy: {train_accuracy*100:.4f}%")
print("\nRun ID: ")
print(f"{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}")
  

Configuration saved as: config_07-10_01-42--00_p5_mlp_equal_convnextv2_tiny.fcmae_ft_in22k_in1k.py in /home/takayuki/Desktop/summer2025/plants/plant_classification/../saved_configs
Starting training...


Epoch 1/100: 100%|██████████| 33/33 [00:13<00:00,  2.45batch/s]


Training Loss: 2.7659, Training Accuracy: 21.9697% at LR: 0.000500
Validation Loss: 2.3433, Validation Accuracy: 36.3636%
Best Validation Accuracy: 36.3636% at Epoch 1
Saved new best model: /home/takayuki/Desktop/summer2025/plants/plant_classification/../training/phase5/models/best_model_07-10_01-42--00_p5_mlp_equal_convnextv2_tiny.fcmae_ft_in22k_in1k.pth
Best Validation Loss: 2.3433 at Epoch 1


Epoch 2/100: 100%|██████████| 33/33 [00:13<00:00,  2.50batch/s]


Training Loss: 1.9371, Training Accuracy: 56.8182% at LR: 0.000500
Validation Loss: 1.7335, Validation Accuracy: 52.2727%
Best Validation Accuracy: 52.2727% at Epoch 2
Saved new best model: /home/takayuki/Desktop/summer2025/plants/plant_classification/../training/phase5/models/best_model_07-10_01-42--00_p5_mlp_equal_convnextv2_tiny.fcmae_ft_in22k_in1k.pth
Best Validation Loss: 1.7335 at Epoch 2


...

Epoch 42/100: 100%|██████████| 33/33 [00:13<00:00,  2.39batch/s]


Training Loss: 0.0627, Training Accuracy: 99.2424% at LR: 0.000338
Validation Loss: 0.3366, Validation Accuracy: 86.3636%


Epoch 43/100: 100%|██████████| 33/33 [00:14<00:00,  2.28batch/s]


Training Loss: 0.0820, Training Accuracy: 97.7273% at LR: 0.000331
Validation Loss: 0.2812, Validation Accuracy: 88.6364%
Saved new best model: /home/takayuki/Desktop/summer2025/plants/plant_classification/../training/phase5/models/best_model_07-10_01-42--00_p5_mlp_equal_convnextv2_tiny.fcmae_ft_in22k_in1k.pth
Best Validation Loss: 0.2812 at Epoch 43


Epoch 44/100:  58%|█████▊    | 19/33 [00:09<00:07,  1.92batch/s]

In [ ]:
# Save the final model
final_model_name = f"final_model_{cfg.train_timm_model_name}_{timestamp}.pth"
final_model_file_path = os.path.join(cfg.models_path, final_model_name)
save_model_weights(model, final_model_file_path)

# check if there is a varaible elapsed_time
try:
    elapsed_time
except NameError:
    end_time = time.time()
    elapsed_time = end_time - start_time

print("\nTraining complete.")
print(f"Best Validation Accuracy: {best_val_accuracy*100:.4f}% at Epoch {best_val_epoch}")
print(f"Final Training Accuracy: {train_accuracy*100:.4f}%")
print("\nRun ID: ")
print(f"{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}")

## Evaluation script

#### Some evaluation metrics
- Accuracy: correct predictions / total predictions
- Precision: TP / (TP + FP)
- Recall: TP / (TP + FN)
- F1 Score: Harmonic mean of precision and recall

In [ ]:
# In a new cell at the end of your training notebook

from utils.evaluate import EvalClassification
from utils.model import load_model # Make sure to import your new loader

# --- 1. Define an Evaluation Helper Function ---
# This avoids code repetition and makes the process clear.
def evaluate_model(model_path, test_loader, hand_loader):
    """
    Loads a model from a path and evaluates it on two different dataloaders.
    
    Returns:
        A dictionary containing all the calculated metrics.
    """
    print(f"\n--- Evaluating model: {os.path.basename(model_path)} ---")
    
    # Load the model robustly using your new function
    eval_model = load_model(model_path, verbose=False)
    
    # === Evaluate on the standard test set ===
    test_evaluator = EvalClassification(model=eval_model, dataloader=test_loader)
    test_evaluator.evaluate()
    test_accuracy = test_evaluator.get_accuracy(verbose=False)
    test_metrics = test_evaluator.get_binary_metrics(display=False)
    
    # === Evaluate on the hand-labeled test set ===
    hand_evaluator = EvalClassification(model=eval_model, dataloader=hand_loader)
    hand_evaluator.evaluate()
    hand_accuracy = hand_evaluator.get_accuracy(verbose=False)
    hand_metrics = hand_evaluator.get_binary_metrics(display=False)
    
    return {
        "test_accuracy": test_accuracy,
        "test_fnr": test_metrics.get('FNR', 0.0),
        "hand_accuracy": hand_accuracy,
        "hand_fnr": hand_metrics.get('FNR', 0.0),
    }

# --- 2. Run Evaluations ---
# Define the paths to your saved models
# Assumes `best_model_path` and `final_model_path` were defined at the end of the training loop
hand_test_loader = data_utils.get_test_dataloader() # Create the hand dataloader

# Evaluate the model that had the best validation accuracy
best_model_metrics = evaluate_model(best_model_file_path, test_dataloader, hand_test_loader)

# Evaluate the final model from the last epoch
final_model_metrics = evaluate_model(final_model_file_path, test_dataloader, hand_test_loader)


In [ ]:
print("\n" + "="*50)
print("              FINAL RESULTS SUMMARY")
print("="*50)
print(f"\nBest Model (Epoch {best_val_epoch}, Val Acc: {best_val_accuracy:.4f}):")
print(f"  - Test Set Accuracy:      {best_model_metrics['test_accuracy']:.4f}")
print(f"  - Test Set FNR:           {best_model_metrics['test_fnr']:.4f}")
print(f"  - Hand-Labeled Accuracy:  {best_model_metrics['hand_accuracy']:.4f}")
print(f"  - Hand-Labeled FNR:       {best_model_metrics['hand_fnr']:.4f}")

print(f"\nFinal Model (Epoch {epoch+1}):")
print(f"  - Test Set Accuracy:      {final_model_metrics['test_accuracy']:.4f}")
print(f"  - Test Set FNR:           {final_model_metrics['test_fnr']:.4f}")
print(f"  - Hand-Labeled Accuracy:  {final_model_metrics['hand_accuracy']:.4f}")
print(f"  - Hand-Labeled FNR:       {final_model_metrics['hand_fnr']:.4f}")
print("="*50)


### Hyperparameters get logged to tensorboard only if the following block is executed after noting observations

In [ ]:
observations = "linear achieves same overall accuracy, but FNR = 10, was 0 for gated attn"

hparams_dict["observations"] = observations
writer.add_text('Observations', observations, 0)

# 2. Calculate total elapsed time
elapsed_time_minutes = (time.time() - start_time) / 60

final_metrics_to_log = {
    
    "best_model/test_accuracy": best_model_metrics['test_accuracy'],
    "best_model/test_fnr": best_model_metrics['test_fnr'],
    "best_model/hand_accuracy": best_model_metrics['hand_accuracy'],
    "best_model/hand_fnr": best_model_metrics['hand_fnr'],
    
    "final_model/test_accuracy": final_model_metrics['test_accuracy'],
    "final_model/test_fnr": final_model_metrics['test_fnr'],
    "final_model/hand_accuracy": final_model_metrics['hand_accuracy'],
    "final_model/hand_fnr": final_model_metrics['hand_fnr'],
    
    "indicators/best_val_accuracy": best_val_accuracy,
    "indicators/best_val_loss": best_val_loss,
    "indicators/best_val_epoch": best_val_epoch,
    "indicators/final_val_accuracy": val_accuracy,
    "indicators/final_train_accuracy": train_accuracy,
    "indicators/final_train_loss": train_loss,
    "indicators/final_val_loss": val_loss,
    "indicators/final_learning_rate": current_lr,
    
    "meta/epochs_trained": epoch + 1,
    "meta/time_elapsed_minutes": elapsed_time_minutes
}

# 4. Write to TensorBoard and close the writer
# Pass the pure hparams and the pure metrics separately.
writer.add_hparams(hparams_dict, final_metrics_to_log)
writer.flush()
writer.close()

print("\nAll metrics and hyperparameters logged correctly to TensorBoard.")